# DataMate — agent code-gen trên dev (D20)

Chạy agent sinh code (`src/datamate/agent/`) với một model ≤ 15B trên **câu dev của train**, ghi replay + trace để lead calibrate trên CPU (`eval/calibrate_agent.py`).

**Yêu cầu** (xem `docs/KAGGLE.md`):
- Accelerator **GPU T4 x2**, **Internet: On**, notebook để **Private**.
- Dataset Kaggle **private** chứa thư mục `train` (có `questions.jsonl`, `answers.jsonl`, `rulebook.md`, `calendar.csv`, `data/`). **Chỉ train**, không đưa public_test lên.

**Đầu ra**: `/kaggle/working/datamate_dev_<model>.zip` (replay, trace, report, log). Tải về máy, **không commit lên GitHub** (chứa câu hỏi train + output model).

In [ ]:
# ---- cấu hình ----
REPO = "https://github.com/dinhkhoi124/DateMate.git"
BRANCH = "feat/llm-parser"
MODEL_REPO = "Qwen/Qwen3-14B-AWQ"      # ứng viên khác: Qwen/Qwen3-8B-AWQ, Qwen/Qwen2.5-Coder-14B-Instruct-AWQ
PER_SHOP = 6                           # 24 shop dev x 6 = 144 câu. Tăng dần khi đã ổn
SAMPLES = 5                            # k mẫu / câu
MAX_LEN = 20480                        # prompt ~10-12k token
VLLM_VERSION = "0.10.1.1"              # bản pin của image (D17); không chạy được trên T4 thì dùng fallback
VLLM_FALLBACK = "0.8.5.post1"
RUN_SMOKE = True                       # 2 câu x 1 shop trước (tốn thêm 1 lần load model)
RUN_HOLDOUT = False                    # chỉ số tổng, không xem lỗi (CLAUDE.md luật 2)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
import subprocess
import sys
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-1500:])
    return r.returncode

def vllm_ok():
    code = "import torch, vllm; assert torch.cuda.is_available(); print('torch', torch.__version__, 'vllm', vllm.__version__)"
    return subprocess.run([sys.executable, "-c", code]).returncode == 0

sh(f"pip install -q vllm=={VLLM_VERSION}")
if not vllm_ok():
    print(f"vLLM {VLLM_VERSION} không chạy được trên máy này -> thử {VLLM_FALLBACK}")
    sh(f"pip install -q vllm=={VLLM_FALLBACK}")
    assert vllm_ok(), "vLLM không chạy được: gửi output cell này cho lead"

In [ ]:
import os
import glob
!rm -rf /kaggle/working/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/working/repo
os.chdir("/kaggle/working/repo")
!git log --oneline -1
# deps runtime giống image (docker/requirements.txt), giữ nguyên torch/vllm vừa cài
!pip install -q -r docker/requirements.txt 2>&1 | tail -3
!python -c "import pandas, numpy; print('pandas', pandas.__version__, 'numpy', numpy.__version__)"

In [ ]:
# dataset train -> data/train/train (repo đọc đường dẫn này)
hits = [p for p in glob.glob("/kaggle/input/**/questions.jsonl", recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), "answers.jsonl"))]
assert hits, "không thấy thư mục train (questions.jsonl + answers.jsonl) trong /kaggle/input"
TRAIN = os.path.dirname(sorted(hits)[0])
print("train:", TRAIN)
os.makedirs("data/train", exist_ok=True)
if not os.path.exists("data/train/train"):
    os.symlink(TRAIN, "data/train/train")
!ls data/train/train | head

In [ ]:
from huggingface_hub import snapshot_download
MODEL_NAME = MODEL_REPO.split("/")[-1]
MODEL_DIR = f"/tmp/models/{MODEL_NAME}"          # tên thư mục = model_id trong khoá replay
snapshot_download(MODEL_REPO, local_dir=MODEL_DIR)
!du -sh {MODEL_DIR}

In [ ]:
import torch
NGPU = torch.cuda.device_count()
OUT = f"/kaggle/working/out_{MODEL_NAME}"
os.makedirs(OUT, exist_ok=True)
os.environ.update({
    "PYTHONPATH": "src", "PYTHONHASHSEED": "0",
    "DATAMATE_LLM_DTYPE": "float16",            # T4 không có bf16; AWQ chạy fp16
    "DATAMATE_LLM_TP": str(min(NGPU, 2)),
    "DATAMATE_LLM_MAX_LEN": str(MAX_LEN),
    "DATAMATE_LLM_GPU_UTIL": "0.90",
    "DATAMATE_LLM_MODEL_ID": MODEL_NAME,
    "DATAMATE_AGENT_SAMPLES": str(SAMPLES),
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
})
print("GPUs:", NGPU, {k: v for k, v in os.environ.items() if k.startswith("DATAMATE_")})
EVAL = f"python eval/run_eval.py --agent codegen --llm vllm --llm-model {MODEL_DIR} --llm-replay {OUT}/replay.jsonl --trace --by source"

In [ ]:
# smoke: 2 câu của 1 shop dev (biz_101, không phải holdout)
if RUN_SMOKE:
    !{EVAL} --split dev --only biz_101 --per-shop 2 2>&1 | tail -25

In [ ]:
# chạy chính trên dev
import time
t0 = time.time()
!{EVAL} --split dev --per-shop {PER_SHOP} 2>&1 | tee {OUT}/run_eval_dev.txt | grep -vE "^(INFO|WARNING|\[)" | tail -40
print(f"{(time.time() - t0) / 60:.1f} phút")

In [ ]:
# calibrate k/m/policy trên trace vừa ghi
DEV_RUN = sorted(glob.glob("eval/runs/*_dev"))[-1]
print(DEV_RUN)
!python eval/calibrate_agent.py {DEV_RUN}/trace.jsonl 2>&1 | tee {OUT}/calibrate_dev.txt

In [ ]:
# (tuỳ chọn) holdout: chỉ số tổng
if RUN_HOLDOUT:
    !{EVAL} --split holdout --per-shop {PER_SHOP} 2>&1 | tee {OUT}/run_eval_holdout.txt | grep -E "^split=" 
    HO_RUN = sorted(glob.glob("eval/runs/*_holdout"))[-1]
    !python eval/calibrate_agent.py {HO_RUN}/trace.jsonl --totals-only 2>&1 | tee {OUT}/calibrate_holdout.txt

In [ ]:
# đóng gói: tải file zip này về máy (Output -> Download). KHÔNG commit lên GitHub.
import shutil
for d in sorted(glob.glob("eval/runs/*_dev")) + sorted(glob.glob("eval/runs/*_holdout")):
    shutil.copytree(d, f"{OUT}/{os.path.basename(d)}", dirs_exist_ok=True)
!pip freeze | grep -iE "^(vllm|torch|transformers|xgrammar)==" > {OUT}/versions.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader >> {OUT}/versions.txt
!git rev-parse --short HEAD >> {OUT}/versions.txt
zipf = shutil.make_archive(f"/kaggle/working/datamate_dev_{MODEL_NAME}", "zip", OUT)
print(zipf, os.path.getsize(zipf) // 1024, "KB")